## Fact Trip Stop 

In [0]:
import dlt
import pyspark.sql.functions as F

In [0]:
%sql

select * from travel_journal_catalog.silver.trip_stops;

address,category,created_at,duration,flag,google_maps_address_id,hours,id,image_id,lat,lng,location_title,rating,review_count,status,time,trip_post_id,user_id,date_type,year,month,day,duration_minutes,open_time,close_time,hours_valid,is_open
"Japan, 〒451-0051 Aichi, Nagoya, Nishi Ward, Noritakeshinmachi, 4 Chome−1−35 産業技術記念館内",tourist attraction,2026-06-24T06:15:34.568296Z,2 hr,false,null,09:00 - 18:00,34,55,null,null,Toyota Commemorative Museum of Industry and Technology トヨタ産業技術記念館,4.5,100,Open,2026-06-24T09:00:00Z,30,16,2026-06-24,2026,6,24,120.0,09:00,18:00,true,true
"1-1 Honmaru, Naka Ward, Nagoya, Aichi 460-0031, Japan",tourist attraction,2026-06-24T06:25:29.495574Z,1.5 hr,false,9,Wednesday: 9:00 AM – 4:00 PM,35,56,35.1848636,136.899927,Honmaru Palace,4.3,3236,Open,2026-06-24T09:00:00Z,31,16,2026-06-24,2026,6,24,90.0,,,null,true
"2-chōme-1-1 Hamachō, Funabashi, Chiba 273-8530, Japan",shopping mall,2026-07-21T04:01:48.439932Z,30 mins,false,880,Tuesday: 10:00 AM – 8:00 PM,906,413,35.6872736,139.9877584,LaLaport TOKYO-BAY,4.1,16319,Open,2026-07-20T23:00:00Z,322,18,2026-07-21,2026,7,21,30.0,,,null,true
"36098 Hughes Cliff Suite 475, Prague, Czech Republic",snowboarding,2026-07-21T04:03:13.448817Z,2 hrs,false,886,08:00 - 16:15,907,416,50.08804,14.42076,Old Heath Street,4.2,349,Open,2026-07-21T09:00:13.221382Z,323,76,2026-07-21,2026,7,21,120.0,08:00,16:15,true,true
"822 Schaefer Valleys Apt. 748, Stuttgart, Germany",concert,2026-07-21T04:03:13.574351Z,30 mins,false,897,19:15 - 08:30,908,414,48.78232,9.17702,Grand Knox Villa,3.5,145,Open,2026-07-21T10:00:13.340926Z,323,76,2026-07-21,2026,7,21,30.0,19:15,08:30,false,true
"611 Debra Branch Apt. 654, Muang Phônsavan, Laos",diving,2026-07-21T04:03:13.695774Z,2 hrs,false,891,11:00 - 16:30,909,415,19.4494,103.1917,Little Reynolds Palace,3.1,35,Closed,2026-07-21T11:00:13.465452Z,323,76,2026-07-21,2026,7,21,120.0,11:00,16:30,true,false
"822 Schaefer Valleys Apt. 748, Stuttgart, Germany",photography,2026-07-21T04:03:13.815852Z,2 hrs,false,897,11:45 - 09:00,910,414,48.78232,9.17702,Grand Knox Villa,3.1,15,Closed,2026-07-21T09:00:13.5858Z,324,76,2026-07-21,2026,7,21,120.0,11:45,09:00,false,false
"66044 Rowland Streets, Sydney, Australia",yoga,2026-07-21T04:03:13.935563Z,2 hrs,false,898,17:15 - 14:15,911,415,-33.86785,151.20732,Little Johnson Park,4.1,460,Open,2026-07-21T10:00:13.706096Z,324,76,2026-07-21,2026,7,21,120.0,17:15,14:15,false,true
"42675 Dalton Lodge Apt. 970, Göteborg, Sweden",mountaineering,2026-07-21T04:03:14.056895Z,1 hr,false,890,08:30 - 12:30,912,416,57.70716,11.96679,Little Pham Gate,4.8,210,Closed,2026-07-21T11:00:13.825063Z,324,76,2026-07-21,2026,7,21,60.0,08:30,12:30,true,false
"611 Debra Branch Apt. 654, Muang Phônsavan, Laos",drawing,2026-07-21T04:03:14.175301Z,2 hrs,false,891,10:45 - 11:15,913,419,19.4494,103.1917,Little Reynolds Palace,3.5,452,Open,2026-07-21T09:00:13.946365Z,325,77,2026-07-21,2026,7,21,120.0,10:45,11:15,true,true


In [0]:
# Expectations 
rules = {
    "valid_id": "id IS NOT NULL",
    "valid_time": "time IS NOT NULL",
}

In [0]:
@dlt.table(name="FactTripStop_stage")

@dlt.expect_all_or_drop(rules)
# This function will do above this source
def FactTripStop_stage():
    df = spark.readStream.table("travel_journal_catalog.silver.trip_stops")
    return df

In [0]:
import dlt
import pyspark.sql.functions as F


@dlt.table(name="fact_trip_stop", table_properties={"quality": "gold"})
@dlt.expect_or_drop("valid_stop_id", "trip_stop_id IS NOT NULL")
def fact_trip_stop():
    stops = (
        dlt.read_stream("FactTripStop_stage")
        .withColumn("stop_count", F.lit(1))
        .withColumnRenamed("flag", "is_flagged")
        .alias("stops_df")
    )

    dim_users = dlt.read("travel_journal_catalog.gold.dim_user").alias("user")
    dim_address = dlt.read("travel_journal_catalog.gold.dimgoogleaddress").alias("google_address")

    # SCD2 point-in-time lookup of the user version
    with_user = stops.join(
        dim_users,
        (F.col("stops_df.user_id") == F.col("user.user_id"))
        & (F.col("stops_df.time") >= F.col("user.__START_AT"))
        & ((F.col("stops_df.time") <= F.col("user.__END_AT")) | F.col("user.__END_AT").isNull()),
        "inner",
    )

    # surrogate-key lookup of the place (left join: keep stops even if unlinked)
    fact = with_user.join(
        dim_address,
        F.col("stops_df.google_maps_address_id") == F.col("google_address.id"),
        how="left",
    ).select(
        F.col("stops_df.id").alias("trip_stop_id"),
        F.col("stops_df.trip_post_id").alias("trip_post_id"),
        F.col("user.user_id").alias("user_id"),
        F.col("user.__START_AT").alias("user_version_at"),
        F.coalesce(F.col("google_address.DimGoogleAddressKey"), F.lit(-1)).alias("google_maps_address_key"),
        F.col("stops_df.time").alias("trip_stop_time"),
        F.col("google_maps_address_id"),
        F.col("stops_df.duration_minutes").alias("trip_stop_duration"),
        F.col("stops_df.rating").alias("rating"),
        F.col("stops_df.stop_count").alias("stop_count"),
        F.col("stops_df.is_flagged").alias("is_flagged"),
    )

    return fact